# Examine drop-out risks and Intervention factors
We will use a combination of 
1. XGBoost global feature importance: how the trees were built
2. Permutation importance: how much performance depends on a feature
3. SHAP global summary: how features contribute to predictions

and compare the results to derive insights on drop-out risks and intervention factors. We combine three complementary methods to ensure robustness and validity of our findings.

Note that none of these methods provide causal insights, but they can help *identify important factors* associated with student drop-out.




In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings('ignore')
import os
from pathlib import Path
import joblib  # // Added because the model was saved with joblib.dump and must be loaded with joblib.load for reliability
from sklearn.preprocessing import MinMaxScaler

print("Libraries imported successfully!")

In [ ]:
# Load the tuned model
current_user = os.getenv('USER')
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
final_model = joblib.load(PROCESSED_DIR / "xgb_final_tuned_trainval.pkl")  
# // Changed from pd.read_pickle to joblib.load because the model was saved with joblib.dump and must be loaded with the same mechanism to avoid silent incompatibilities
tuning_outputs = joblib.load(PROCESSED_DIR / "xgb_tuning_outputs.pkl")
# // Load saved training metadata so feature importance can be aligned to the exact feature schema used during model training

print(f"Model loaded successfully!")


In [ ]:
# Schema integrity checks (run once after loading)

# // Retrieve the exact feature schema used during model training to validate downstream importance and SHAP analysis
feature_cols = tuning_outputs["feature_cols"]

# Check that the model exposes a booster and supports probability prediction
# // Ensures the loaded artifact is a tree-based XGBoost model required for gain-based importance
assert hasattr(final_model, "get_booster"), "Loaded object does not appear to be an XGBoost model"

# // Confirms the model is suitable for probability-based evaluation and SHAP analysis
assert hasattr(final_model, "predict_proba"), "Loaded model does not support predict_proba"

# Verify feature count consistency between saved schema and model
booster = final_model.get_booster()
booster_features = booster.feature_names

# // Ensures feature names were preserved during training, which is required for correct importance mapping
assert booster_features is not None, "Booster feature names are missing"

# // Detects silent mismatches between training schema and model internals that would corrupt explanations
assert len(booster_features) == len(feature_cols), (
    f"Feature count mismatch: booster={len(booster_features)}, schema={len(feature_cols)}"
)

# Optional visibility check (safe to print once)
print(f"Schema check passed: {len(feature_cols)} features aligned between model and metadata")

## 1. XGBoost Global Feature Importance

// Old comment

Global feature importance shows which features are most important for the model's decisions across all predictions. We'll use the 'gain' metric, which measures the average improvement in model performance when a feature is used for splitting.

// New comment

## 1. XGBoost Global Feature Importance
Global feature importance provides an aggregated view of which features the XGBoost model relied on most during tree construction across all predictions. We use the (total) gain metric, which measures the total reduction in the model’s training loss contributed by splits on a feature, summed across all trees. This offers a structural, model-level perspective on feature usage, rather than a direct explanation of individual predictions.

Additional Comments

-The original implementation relied on the gain metric, which can represent an average per split depending on the XGBoost version. For global importance, total gain is more appropriate because it reflects the full contribution of a feature across all trees rather than its average impact per split.

-Raw gain, cover, and weight values were plotted without normalization, which can lead to misleading visual comparisons. These metrics exist on different scales and should not be compared directly unless converted to relative (percentage) importance.

-The weight metric was treated as a feature importance measure, even though it only reflects split frequency. This metric is biased toward high-cardinality or continuous features and is best interpreted as a diagnostic rather than as true importance.

-Features that were never used in any tree split were omitted entirely by get_score(), which can give the false impression that such features were not part of the model input. For transparency, unused features should be retained with zero importance.

-Finally, gain-based importance reflects how the model was constructed, not how features contribute to predictions. As such, it should be interpreted as complementary to, rather than interchangeable with, SHAP-based explanations.

In [ ]:
# Extract XGBoost global feature importance
# Get importance scores for different metrics
importance_gain = final_model.get_booster().get_score(importance_type='total_gain')  # // Use total_gain to reflect total loss reduction across all splits (more appropriate for "global" importance than per-split averages)
importance_weight = final_model.get_booster().get_score(importance_type='weight')    # // Split frequency, only as a diagnostic (split frequency is biased and not a reliable "importance" measure)
importance_cover = final_model.get_booster().get_score(importance_type='total_cover')# // Use total_cover to reflect total sample coverage across all splits rather than an average per split

# Convert to DataFrame for easier analysis
def importance_to_df(importance_dict, importance_type, all_features):
    df = pd.DataFrame(list(importance_dict.items()),
                     columns=['feature', f'importance_{importance_type}'])
    df = df.set_index('feature').reindex(all_features).fillna(0.0).reset_index()     # // Ensure unused features are included with 0 importance so results reflect the full model input space
    total = df[f'importance_{importance_type}'].sum()
    df[f'importance_{importance_type}_pct'] = (100 * df[f'importance_{importance_type}'] / total) if total > 0 else 0.0  # // Normalize to % so importances are interpretable and comparable within a metric
    df = df.sort_values(f'importance_{importance_type}', ascending=False)
    return df

all_features = list(tuning_outputs["feature_cols"])  # // Use the saved training feature schema to prevent silent feature-order/name mismatches in importance reporting

df_gain = importance_to_df(importance_gain, 'total_gain', all_features)
df_weight = importance_to_df(importance_weight, 'weight', all_features)
df_cover = importance_to_df(importance_cover, 'total_cover', all_features)

Comment: 

We report XGBoost’s global feature importance using the booster’s gain-based importance (total_gain), 

which aggregates the total reduction in the training objective attributable to splits on each feature across all trees; 

importances are normalized (%) and unused features are retained with zero importance for transparency. 

For contextual interpretation, we additionally report total_cover, which reflects the total number of training samples affected by splits on a feature, 

and weight, which reflects split frequency across all trees and is included as a diagnostic indicator rather than as a measure of predictive contribution.

In [ ]:
# Visualize XGBoost global feature importance
plt.figure(figsize=(12, 12))

# Plot top 20 features by total gain
top_20_gain = df_gain.head(20)
plt.subplot(3, 1, 1)

# // Use normalized total_gain (%) so values are interpretable and comparable within this metric
plt.barh(range(len(top_20_gain)), top_20_gain['importance_total_gain_pct'])

plt.yticks(range(len(top_20_gain)), top_20_gain['feature'])

# // Label reflects that importance is expressed as percentage contribution to total training-loss reduction
plt.xlabel('Importance (% of total gain)')

# // Clarify that total_gain (not average gain) is used for global importance
plt.title('Top 20 Features - XGBoost Global Feature Importance (Total Gain)')

plt.gca().invert_yaxis()

# Plot top 20 features by weight (diagnostic only)
top_20_weight = df_weight.head(20)
plt.subplot(3, 1, 2)

# // Normalize split frequency to percentage to avoid misleading raw-count comparisons
plt.barh(range(len(top_20_weight)), top_20_weight['importance_weight_pct'])

plt.yticks(range(len(top_20_weight)), top_20_weight['feature'])

# // Explicitly label weight as split frequency, not predictive importance
plt.xlabel('Split Frequency (% of total)')

# // Mark weight as diagnostic to avoid over-interpreting it as true importance
plt.title('Top 20 Features - XGBoost Split Frequency (Weight, Diagnostic)')

plt.gca().invert_yaxis()

# Plot top 20 features by total cover
top_20_cover = df_cover.head(20)
plt.subplot(3, 1, 3)

# // Use normalized total_cover (%) to reflect aggregate sample coverage across all splits
plt.barh(range(len(top_20_cover)), top_20_cover['importance_total_cover_pct'])

plt.yticks(range(len(top_20_cover)), top_20_cover['feature'])

# // Clarifies that cover reflects how many samples are affected by splits on a feature
plt.xlabel('Sample Coverage (% of total)')

plt.title('Top 20 Features - XGBoost Global Feature Importance (Total Cover)')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
# Print top 20 features for each global importance metric

print("\nTop 20 Features by Total Gain (% of total):")
# // gain
display(df_gain[['feature', 'importance_total_gain_pct']].head(20))

print("\nTop 20 Features by Weight (Split Frequency %, Diagnostic):")
# // weight
display(df_weight[['feature', 'importance_weight_pct']].head(20))

print("\nTop 20 Features by Total Cover (% of total):")
# // cover
display(df_cover[['feature', 'importance_total_cover_pct']].head(20))


## 1.1 Results & interpretation

The XGBoost global feature importance analysis shows that Block A academic performance is the dominant driver of dropout predictions. Average Grade A and Total Credits Block A account for the largest share of total gain, indicating that early academic outcomes play a central role in distinguishing between drop-outs and non-drop-outs. Timing- and distance-related features contribute additional structure by refining decision boundaries and segmenting students into subgroups. Degree-related features primarily operate at broader splits, suggesting program-level differences in dropout patterns.

## 2. Permutation Importance

Permutation importance measures how much the model's performance (Average Precision/PR-AUC) decreases when each feature is randomly shuffled. This approach is more robust than tree-based feature importance because:
1. It's model-agnostic and based on actual prediction performance
2. It's less biased toward high-cardinality categorical features
3. It captures feature interactions better

//Rephrased: 

Permutation importance measures how much the model's performance (Average Precision/PR-AUC) decreases when each feature is randomly shuffled. This approach is complementary to tree-based feature importance because:

1. It's model-agnostic and based on actual prediction performance

2. It's less biased toward high-cardinality categorical features

3. It reflects the importance of a feature for predictive performance, including cases where effects arise through interactions with other features.

permutation importance does not explicitly model interactions,it implicitly reflects interactions only insofar as breaking one feature disrupts joint effects


In [ ]:
# Perform permutation importance analysis
# Load the test data first
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

# Define target and feature columns
target_col = 'sdo5_degree_drop_out'
exclude_cols = ['set', target_col]

# // Use the saved training feature schema to ensure permutation importance is computed on the exact feature space the model was trained on
feature_cols = tuning_outputs["feature_cols"]

# Extract test set
test_df = df[df['set'] == 'test'].copy()

# // Enforce column order and feature alignment to prevent silent corruption of permutation importance scores
X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

# Perform permutation importance on test set
# Using multiple repetitions for more robust estimates
perm_importance = permutation_importance(
    final_model, 
    X_test, 
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring='average_precision'  # Using same metric as model tuning
)

# Create DataFrame with results
perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)


In [ ]:
# Visualize permutation importance
plt.figure(figsize=(12, 8))

# Plot top 20 features
top_20_perm = perm_df.head(20)
plt.barh(
    range(len(top_20_perm)),
    top_20_perm['importance_mean'],
    xerr=top_20_perm['importance_std']
)
plt.yticks(range(len(top_20_perm)), top_20_perm['feature'])

# // Shortened label to emphasize that values represent performance drop, not absolute performance
plt.xlabel('Decrease in Average Precision (PR-AUC)')

# // Slightly simplified title to keep focus on evaluation set and method
plt.title('Top 20 Features - Permutation Importance (Test Set)')

# // Add reference line to distinguish features with positive vs near-zero/negative impact on performance
plt.axvline(0, linestyle='--', linewidth=1)

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Print top 20 features by permutation importance

print("\nTop 20 Features by Permutation Importance (PR-AUC decrease):")
display(
    perm_df[['feature', 'importance_mean', 'importance_std']]
    .head(20)
)


## 2.1 Results & interpretation

All Block A course result features rank among the most important variables under permutation importance, confirming the results from the XGBoost gain-based analysis. In particular, `Total Credits Block A` and `Average Grade A` cause the largest decreases in PR-AUC when permuted, indicating that early `academic performance` is central to the model’s predictive capability. In addition, features such as `has_dsa` and gender also contribute to predictive performance, although to a lesser extent than academic outcomes. This suggests that while dropout risk is primarily driven by early performance, structural and demographic factors provide complementary information.

## 3. SHAP Global Summary

The goal is to create a global summary showing the average impact of each feature. SHAP (SHapley Additive exPlanations) values provide a unified framework for interpreting model predictions. SHAP values reflect contributions to the model’s output (log-odds) for dropout. SHAP answers the questions: 

Which features contributed the most to predictions?

Why did the model give this specific prediction?

why SHAP?

-Exact for tree models (TreeSHAP)

-Works globally and locally

-Additive → contributions sum to the prediction

-Consistent feature attribution (theoretical guarantee)


In [ ]:
# SHAP Analysis
# Create SHAP explainer for XGBoost
explainer = shap.TreeExplainer(final_model)

# Calculate SHAP values for test set (sample if too large for memory)
# Using a sample for computational efficiency
sample_size = min(5000, len(X_test))
X_test_sample = X_test.sample(n=sample_size, random_state=42)
y_test_sample = y_test.loc[X_test_sample.index]

print(f"Calculating SHAP values for {sample_size} test samples...")
shap_values = explainer.shap_values(X_test_sample)
print("SHAP values calculated successfully!")

# Global feature importance based on mean absolute SHAP values
shap_importance = pd.DataFrame({
    'feature': X_test_sample.columns,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print("Top 20 features by SHAP importance:")
print(shap_importance.head(20))

In [ ]:
# Standalone SHAP Summary Plot for detailed analysis
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, max_display=20, show=False)
plt.title('SHAP Summary Plot - Feature Impact Distribution\n(Red = High feature values, Blue = Low feature values)', 
          fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## 3.1 SHAP Results & Interpretation

SHAP values provide the most detailed and theoretically sound explanation of our model's predictions. Each SHAP value represents the contribution of a specific feature to moving the prediction away from the expected baseline.

Key insights from the SHAP analysis:

**Most Important Features (by mean absolute SHAP value):**
- Similar to previous methods, Block A academic performance features dominate
- The SHAP summary plot shows both the importance and the directional impact of features
- Positive SHAP values push predictions toward dropout, while negative SHAP values push toward retention; color indicates feature value (red = high, blue = low)

**Feature Impact Patterns:**
- **Total Credits Block A**: Lower credits strongly associated with higher dropout risk
- **Average Grade A**: Lower grades associated with higher dropout risk  
- **Course result flags**: Missing course results are significant risk factors

**SHAP Advantages:**
1. **Directional insights**: Unlike other methods, SHAP shows whether high/low feature values increase or decrease dropout risk
2. **Individual-level explanations**: SHAP enables individual-level explanations, although this analysis focuses on global patterns 
3. **Interaction effects**: While SHAP supports interaction analysis, this study focuses on main feature effects
4. **Theoretical soundness**: Based on cooperative game theory with desirable mathematical properties

## 4. Comparative Analysis

Let's compare the results from all three methods to identify the most robust dropout risk factors.

In [ ]:
# // Weight-based importance is included for diagnostic purposes
# // to illustrate how frequently features are used during tree construction.
# // It is excluded from the aggregated importance score because it does not
# // measure predictive contribution and is not directly comparable to gain,
# // permutation importance, or SHAP values.

# Comparative analysis of all three methods
# Combine results from all three approaches
comparison_df = pd.DataFrame({
    'feature': feature_cols
})

# Add XGBoost importance (total gain, %)
comparison_df = comparison_df.merge(
    df_gain.rename(columns={'importance_total_gain_pct': 'xgb_total_gain_pct'})[
        ['feature', 'xgb_total_gain_pct']
    ],
    on='feature', how='left'
).fillna(0)

# Add XGBoost importance (weight, split frequency %; diagnostic only)
comparison_df = comparison_df.merge(
    df_weight.rename(columns={'importance_weight_pct': 'xgb_weight_pct'})[
        ['feature', 'xgb_weight_pct']
    ],
    on='feature', how='left'
).fillna(0)

# Add permutation importance
comparison_df = comparison_df.merge(
    perm_df.rename(columns={'importance_mean': 'perm_importance'})[
        ['feature', 'perm_importance']
    ],
    on='feature', how='left'
).fillna(0)

# Add SHAP importance
comparison_df = comparison_df.merge(
    shap_importance[['feature', 'shap_importance']],
    on='feature', how='left'
).fillna(0)

# Normalize importance scores to 0–1 scale for comparison
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

# // Exclude weight from aggregation to ensure the average reflects
# // predictive contribution rather than split frequency
importance_cols = ['xgb_total_gain_pct', 'perm_importance', 'shap_importance']
diagnostic_cols = ['xgb_weight_pct']

comparison_df[importance_cols + diagnostic_cols] = scaler.fit_transform(
    comparison_df[importance_cols + diagnostic_cols]
)

# Calculate average importance across predictive methods only
comparison_df['avg_importance'] = comparison_df[importance_cols].mean(axis=1)
comparison_df = comparison_df.sort_values('avg_importance', ascending=False)

# Display top 20 features across all methods
print("Top 20 Features - Comparative Analysis (Normalized 0–1 Scale):")
print(
    comparison_df.head(20)[
        ['feature'] + importance_cols + diagnostic_cols + ['avg_importance']
    ].round(3)
)


In [ ]:
# Visualize comparative analysis
plt.figure(figsize=(15, 10))

# Heatmap of top 20 features across all methods
top_20_comparison = comparison_df.head(20)

plt.subplot(2, 1, 1)

# // Include weight in the heatmap as a diagnostic row, even though it is excluded from the aggregated score
heatmap_cols = importance_cols + diagnostic_cols

heatmap_data = top_20_comparison[heatmap_cols].values.T
plt.imshow(heatmap_data, cmap='YlOrRd', aspect='auto')
plt.colorbar(label='Normalized Importance Score')

# // Update labels to match the heatmap column order, keeping weight explicitly marked as diagnostic
plt.yticks(
    range(len(heatmap_cols)),
    ['XGBoost (Total Gain %)', 'Permutation', 'SHAP', 'XGBoost (Weight %, Diagnostic)']
)

plt.xticks(
    range(len(top_20_comparison)),
    top_20_comparison['feature'], rotation=45, ha='right'
)
plt.title('Top 20 Features - Importance Heatmap Across Methods')

# Bar plot of average importance
plt.subplot(2, 1, 2)
plt.barh(range(len(top_20_comparison)), top_20_comparison['avg_importance'])
plt.yticks(range(len(top_20_comparison)), top_20_comparison['feature'])
plt.xlabel('Average Normalized Importance Score (Excludes Weight)')
# // Clarify that the aggregated score excludes weight so readers do not assume split frequency influences the average
plt.title('Top 20 Features - Average Importance Across Predictive Methods')
# // Align title with the aggregation definition (gain + permutation + SHAP only)
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# Summary of most consistent important features
print("\nMost Consistent Important Features (appearing in top 10 of multiple methods):")

# // Keep weight in the consistency summary as a diagnostic comparator, but interpret it separately from predictive methods
top_10_per_method = {
    'XGBoost_TotalGain': set(df_gain.head(10)['feature']),
    'Permutation': set(perm_df.head(10)['feature']),
    'SHAP': set(shap_importance.head(10)['feature']),
    'XGBoost_Weight_Diagnostic': set(df_weight.head(10)['feature'])
}

# Find features appearing in top 10 of at least 3 methods
all_features = set()
for method_features in top_10_per_method.values():
    all_features.update(method_features)

consistent_features = []
for feature in all_features:
    count = sum(
        1 for method_features in top_10_per_method.values()
        if feature in method_features
    )
    if count >= 3:
        consistent_features.append((feature, count))

consistent_features.sort(key=lambda x: x[1], reverse=True)

for feature, count in consistent_features:
    print(f"- {feature}: appears in top 10 of {count}/4 methods")
